# 4.0 Final test on 2024 (run once)
**Purpose.** The single run on the locked test year (paper Section 6.2): the three comparisons of Section 5.1 with week-block bootstrap intervals, and the descriptive results of Section 5.2. Nothing is chosen here. Every model, threshold and reference is loaded from `models/` or refitted from it.

**How it is run.** First build the 2024 district series with `uv run python -m madrid_congestion.districts build-district-series 2024 --test-run`, which runs `check_frozen()` first. Then Restart and Run All. The result is reported as it comes out, and the notebook is not rerun to change it.

**Method.**
1. `check_frozen` and `check_references`.
2. Every model refitted on the 2022 usable calm origins from the frozen settings.
3. The 2023 replay: all 11 scores rescored and required to reproduce every frozen 2023 AP and warning threshold, as in notebook 3.6. Only if it passes is 2024 read.
4. Labels, rows, SARIMA paths and scores of 2024.
5. The three comparisons per horizon (`compare`): point estimate on the 52 complete weeks, 95% paired week-block interval and second-seed check.
6. Descriptive results: AP, POD, FAR and CSI at the frozen 2023 thresholds, MAE against the naive forecast, and AP per district without intervals.

**Inputs and outputs.** The district series 2022, 2023 and 2024 and the frozen files in `models/`. The 2024 scores go to `data/interim/scores-2024.parquet`.

## 1. Frozen design and references

In [5]:
import time

import numpy as np
import polars as pl
from sklearn.metrics import average_precision_score

from madrid_congestion.config import HORIZONS, INTERIM_DATA_DIR, MODELS_DIR, PANEL_YEAR, PROCESSED_DATA_DIR, TEST_YEAR, VALIDATION_YEAR
from madrid_congestion.evaluation import forecast_mae, warning_table
from madrid_congestion.features import origin_rows
from madrid_congestion.frozen import (
    FILES,
    SCORES,
    check_frozen,
    check_references,
    compare,
    frozen_models,
    frozen_sarima_paths,
    load_frozen,
    replay_validation,
    score_year,
)
from madrid_congestion.labels import add_labels, year_grid
from madrid_congestion.sarima import sarima_grid

SCORED_YEAR = TEST_YEAR
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

check_frozen()
series = {year: pl.read_parquet(PROCESSED_DATA_DIR / f"district-series-{year}.parquet") for year in (PANEL_YEAR, VALIDATION_YEAR)}
check_references(series[PANEL_YEAR])
refs = pl.read_parquet(MODELS_DIR / FILES["references"])  # from here on, only frozen values are used
calendar = pl.read_parquet(MODELS_DIR / FILES["calendar"])
districts = refs["distrito"]
labels = {year: add_labels(year_grid(df, districts, year), refs) for year, df in series.items()}
rows = {(year, minutes): origin_rows(labels[year], districts, minutes) for year in labels for minutes in HORIZONS}
print("frozen design and 2022 references confirmed |", len(districts), "districts")

frozen design and 2022 references confirmed | 19 districts


## 2. Models refitted from the frozen settings

In [6]:
start = time.time()
models = frozen_models({minutes: rows[PANEL_YEAR, minutes] for minutes in HORIZONS}, districts)
print(f"{len(models)} models refitted in {(time.time() - start) / 60:.1f} minutes")

14 models refitted in 1.3 minutes


## 3. The 2023 replay, before 2024 is read
`replay_validation` stops the run unless every 2023 AP and warning threshold equals its frozen value to 1e-9.

In [7]:
start = time.time()
grid = sarima_grid([labels[PANEL_YEAR], labels[VALIDATION_YEAR]])
paths = frozen_sarima_paths(grid)
replay = replay_validation(
    {minutes: score_year(rows[VALIDATION_YEAR, minutes], minutes, districts, models, calendar, grid, paths) for minutes in HORIZONS}
)
print(f"2023 replay passed in {(time.time() - start) / 60:.1f} minutes | largest AP difference", replay["difference"].max(),
      "| largest threshold difference", replay["threshold_difference"].max())

2023 replay passed in 11.8 minutes | largest AP difference 0.0 | largest threshold difference 0.0


## 4. Scores of the test year
The 2024 labels use the frozen 2022 references. The SARIMA paths run from 1 January 2022 to the end of 2024 with the frozen parameters.

In [9]:
if SCORED_YEAR not in series:
    path = PROCESSED_DATA_DIR / f"district-series-{SCORED_YEAR}.parquet"
    assert path.exists(), f"build it first: build-district-series {SCORED_YEAR} --test-run"
    series[SCORED_YEAR] = pl.read_parquet(path)
    labels[SCORED_YEAR] = add_labels(year_grid(series[SCORED_YEAR], districts, SCORED_YEAR), refs)
    for minutes in HORIZONS:
        rows[SCORED_YEAR, minutes] = origin_rows(labels[SCORED_YEAR], districts, minutes)
    start = time.time()
    grid = sarima_grid([labels[year] for year in sorted(labels)])  # 2022, 2023 and 2024 as one continuous grid
    paths = frozen_sarima_paths(grid)
    print(f"SARIMA paths to the end of {SCORED_YEAR} in {(time.time() - start) / 60:.1f} minutes")
scored = {minutes: score_year(rows[SCORED_YEAR, minutes], minutes, districts, models, calendar, grid, paths) for minutes in HORIZONS}
summary = {}
for minutes, df in scored.items():
    assert df["target"].dtype == pl.Boolean and df.height > 0, minutes  # a non-empty binary target
    target = df["target"].to_numpy()
    summary[minutes] = {"origins": df.height, "positive": int(target.sum()), "base_rate": round(float(target.mean()), 4)}
print(summary)

SARIMA paths to the end of 2024 in 17.5 minutes
{30: {'origins': 474085, 'positive': 9207, 'base_rate': 0.0194}, 60: {'origins': 451313, 'positive': 17917, 'base_rate': 0.0397}}


## 5. The comparisons (paper Section 5.1)
For each horizon, the AP difference on the 52 complete weeks (30 and 31 December outside) and its 95% paired week-block interval. Each difference is read through its sign, its size and the precision of its interval, not as a win or a loss (paper Section 5.4).

In [10]:
start = time.time()
comparisons = compare(scored, SCORED_YEAR)
print(f"bootstrap in {(time.time() - start) / 60:.1f} minutes")
comparisons.with_columns(pl.col("difference", "low", "high", "second_seed_move").round(4))

bootstrap in 13.0 minutes


minutes,comparison,difference,low,high,repetitions,second_seed_move
i64,str,f64,f64,f64,i64,f64
30,"""primary: direct - occupancy""",0.0191,0.0133,0.025,2000,0.0001
30,"""secondary: occupancy - flow""",0.2149,0.1985,0.2305,2000,0.0008
30,"""secondary: with flow - without…",-0.0036,-0.0084,0.0017,2000,0.0002
60,"""primary: direct - occupancy""",0.025,0.0205,0.0292,2000,0.0003
60,"""secondary: occupancy - flow""",0.1527,0.1395,0.166,2000,0.0007
60,"""secondary: with flow - without…",-0.0022,-0.0071,0.003,2000,0.0004


## 6. Descriptive results (paper Section 5.2)
Over every usable calm origin of 2024: AP with the base rate, and POD, FAR and CSI at the frozen 2023 thresholds.

In [11]:
frozen_thresholds = load_frozen(FILES["thresholds"])
measures = pl.concat([
    warning_table(df["target"], {name: df[name] for name in SCORES}, {name: frozen_thresholds[str(minutes)][name]["warning_threshold"] for name in SCORES})
    .select(pl.lit(minutes).alias("minutes"), pl.lit(df["target"].mean()).alias("base_rate"), pl.all())
    for minutes, df in scored.items()
])
measures.with_columns(pl.col("base_rate", "ap", "pod", "far", "csi").round(4), pl.col("warning_threshold").round(6))

minutes,base_rate,score,ap,warning_threshold,pod,far,csi,hits,misses,false_alarms
i32,f64,str,f64,f64,f64,f64,f64,i64,i64,i64
30,0.0194,"""proximity""",0.2617,0.908243,0.4699,0.7198,0.2129,4326,4881,11114
30,0.0194,"""calendar""",0.344,0.191667,0.4491,0.6422,0.2487,4135,5072,7422
30,0.0194,"""linear_occupancy""",0.4278,0.959307,0.5134,0.6086,0.2855,4727,4480,7350
30,0.0194,"""linear_flow""",0.3254,1.028972,0.3755,0.6497,0.2213,3457,5750,6413
30,0.0194,"""linear_direct""",0.5307,0.218178,0.5112,0.5241,0.3271,4707,4500,5184
30,0.0194,"""xgboost_occupancy""",0.6931,1.003616,0.6479,0.3808,0.4633,5965,3242,3669
30,0.0194,"""xgboost_flow""",0.4779,1.022491,0.4805,0.529,0.3121,4424,4783,4968
30,0.0194,"""xgboost_direct""",0.7123,0.283188,0.6795,0.3947,0.4709,6256,2951,4079
30,0.0194,"""xgboost_direct_no_flow""",0.7157,0.299582,0.6764,0.3853,0.475,6228,2979,3904


**Forecast accuracy.** MAE of each flow and occupancy forecast against the realised window maximum, next to the naive forecast (the value at *t*), in relative units and in original units (each value multiplied by its district's frozen reference).

In [12]:
mae = []
for minutes, df in scored.items():
    origins = rows[SCORED_YEAR, minutes]
    for variable, reference in (("occupancy", "threshold"), ("flow", "flow_reference")):
        for family in ("linear", "xgboost", "sarima"):
            mae.append({"minutes": minutes, "forecast": variable, "model": family,
                        **forecast_mae(df[f"{family}_{variable}"], origins[f"{variable}_max_{minutes}"], origins[f"{variable}_lag0"], origins[reference])})
pl.DataFrame(mae).with_columns(pl.exclude("minutes", "forecast", "model").round(4))

minutes,forecast,model,mae_relative,naive_mae_relative,mae_original_units,naive_mae_original_units,mae_reduction
i64,str,str,f64,f64,f64,f64,f64
30,"""occupancy""","""linear""",0.0428,0.053,0.5327,0.6533,0.1931
30,"""occupancy""","""xgboost""",0.032,0.053,0.4043,0.6533,0.3954
30,"""occupancy""","""sarima""",0.0411,0.053,0.5069,0.6533,0.225
30,"""flow""","""linear""",0.0312,0.0387,19.4453,24.0496,0.1939
30,"""flow""","""xgboost""",0.0201,0.0387,12.5695,24.0496,0.4817
30,"""flow""","""sarima""",0.0254,0.0387,15.7805,24.0496,0.344
60,"""occupancy""","""linear""",0.0596,0.0798,0.7377,0.9772,0.2533
60,"""occupancy""","""xgboost""",0.0381,0.0798,0.4779,0.9772,0.5231
60,"""occupancy""","""sarima""",0.0549,0.0798,0.6731,0.9772,0.3122


**Per district (no intervals).** AP of the three XGBoost approaches and the flow-ablation variant in each district, with its base rate. Districts 3 and 9 had pronounced sensor-composition shifts in 2022 (paper Section 3.3).

In [13]:
XGBOOST = ["xgboost_direct", "xgboost_occupancy", "xgboost_flow", "xgboost_direct_no_flow"]
per_district = []
for minutes, df in scored.items():
    for (district,), part in df.group_by("distrito"):
        per_district.append({"minutes": minutes, "distrito": district, "origins": part.height, "base_rate": part["target"].mean(),
                             **{name: float(average_precision_score(part["target"], part[name])) for name in XGBOOST}})
pl.DataFrame(per_district).sort("minutes", "distrito").with_columns(pl.col("base_rate", *XGBOOST).round(4))

minutes,distrito,origins,base_rate,xgboost_direct,xgboost_occupancy,xgboost_flow,xgboost_direct_no_flow
i64,i64,i64,f64,f64,f64,f64,f64
30,1,26397,0.0124,0.2125,0.2179,0.0325,0.2039
30,2,22776,0.0262,0.6847,0.7025,0.422,0.7266
30,3,26543,0.0129,0.6957,0.682,0.5656,0.6905
30,4,25869,0.0231,0.7792,0.7724,0.6203,0.7714
30,5,25549,0.0202,0.8846,0.8337,0.7903,0.8755
30,6,24013,0.0341,0.7579,0.7461,0.6918,0.7488
30,7,26338,0.02,0.548,0.5339,0.3357,0.5412
30,8,22222,0.0271,0.9159,0.9073,0.7747,0.9302
30,9,20880,0.0326,0.7708,0.8036,0.6646,0.7819


## 7. Saved scores

In [14]:
out = pl.concat([df.select(pl.lit(SCORED_YEAR).alias("year"), pl.lit(minutes).alias("minutes"), pl.all()) for minutes, df in scored.items()])
out.write_parquet(INTERIM_DATA_DIR / f"scores-{SCORED_YEAR}.parquet")
print(out.group_by("minutes").len().sort("minutes"))

shape: (2, 2)
┌─────────┬────────┐
│ minutes ┆ len    │
│ ---     ┆ ---    │
│ i32     ┆ u32    │
╞═════════╪════════╡
│ 30      ┆ 474085 │
│ 60      ┆ 451313 │
└─────────┴────────┘


**Result.** The 2023 replay reproduced every frozen AP and warning threshold with a difference of exactly 0 before 2024 was read. The test year holds 474,085 usable calm origins at 30 minutes (base rate 1.94%) and 451,313 at 60 minutes (3.97%).

| Comparison (XGBoost) | ΔAP, 30 min | 95% interval, 30 min | ΔAP, 60 min | 95% interval, 60 min |
|---|---|---|---|---|
| Direct − occupancy | +0.0191 | +0.0133 to +0.0250 | +0.0250 | +0.0205 to +0.0292 |
| Occupancy − flow | +0.2149 | +0.1985 to +0.2305 | +0.1527 | +0.1395 to +0.1660 |
| With flow − without flow | −0.0036 | −0.0084 to +0.0017 | −0.0022 | −0.0071 to +0.0030 |

The second seed moved no endpoint by more than 0.0008, so 2,000 repetitions were kept. The direct classifier has the highest AP (0.712 at 30 minutes and 0.777 at 60 minutes), slightly above the occupancy forecast (0.693 and 0.752), and both intervals of this difference lie above zero. The occupancy forecast is far above the flow forecast (0.478 and 0.598), as expected for an event defined on occupancy. Removing flow gives a slightly higher AP, the opposite sign to 2023, but both intervals include zero. Every XGBoost score lies far above the calendar and proximity baselines. Per district, AP rises with the base rate, and districts 3 and 9 do not stand out in the gap between the occupancy and flow forecasts.

**Conclusion.** For this operational event, direct classification warned best at both horizons, with occupancy forecasting close behind and flow forecasting clearly last. Recent flow added no clear value to the direct classifier.

**Execution record.** The execution counts start at 5 and skip 8. Count 8 was most likely a first attempt of the scoring cell before the 2024 district series existed, which stops at its file check before any data are read. The series was then built with `--test-run`, and the remaining cells ran in order. The notebook was therefore not one uninterrupted Restart and Run All, and it has not been rerun since (paper Section 7.1).